# This is the report template for the Assignment in CSCU9M3 Autumn 2026

## Follow these instructions:
- Save this file in the form: __ZZZZZZZ_M3_Aut26_TheAssignment.ipynb__ where __ZZZZZZZ__ is your student number
- Fill in the report 
 * Use a mix of code and markdown cells to tell a data-story, only the *Executive Summary* should be contained in a single markdown cell
 * all other sections should have multiple cells and outputs, where you explain what you are doing and why.
- When you are finished then ...
 * Save the file
 * Restart the Kernel and re-run the whole notebook. "Kernel" -> "Restart & Run" in the menu at top
 * Make sure everything has executed as expected. If cells run with error that you want to keep to show your attempt, then manually run every cell below the faulty one to show the output.
 * Make sure no __Markdown__ cell is in *edit* mode
 * Save file again
 * Delete this cell and remove any instructions from the template (except headings)
 * __Download this notebook as HTML__, you do that under *File->Download as-> HTML*
   + make sure to use the same name format, just different extension
 * __Upload the html file__ to the assessment on Canvas and submit

## Executive Summary
Write a short (200 words max) summary of what the report contains and what you have discovered.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Introduction
The World Bank's *Gender Statistics* database contains **991 indicators** for **265 economies** back to 1960.
It covers paid work, education, health, fertility, legal rights, financial inclusion and political
representation, and most indicators are published separately for women and for men.

I chose it for three reasons. Almost every indicator has a **female and a male version**, so the natural
unit of analysis is the *gap between women and men inside a country*. The data are a **panel**, so I can
study change over time rather than a single snapshot. And the topic is a global policy target: gender
equality is United Nations Sustainable Development Goal 5.

Going in, I expected gender gaps to be closing steadily as countries became richer and better educated, and
I suspected that girls' education would be the force closing the labour market gap. The report shows that
the first expectation is only partly true, and that the second is not supported by the data at all.

From the 991 indicators I selected **eight** that (a) exist for both sexes, (b) are measured in comparable
units, and (c) together describe four different dimensions of gender inequality: work, health, education and
political voice. Keeping an explicit dictionary of the indicators I use keeps the rest of the notebook
readable, because every later step refers to a short variable name instead of a long World Bank code.

In [2]:
# select 8 indicator from World Bank's Gender Statistics
INDICATORS = {
    "SL.TLF.CACT.FE.ZS": "Female labour force participation rate (% of women aged 15+)",
    "SL.TLF.CACT.MA.ZS": "Male labour force participation rate (% of men aged 15+)",
    "SP.DYN.LE00.FE.IN": "Female life expectancy at birth (years)",
    "SP.DYN.LE00.MA.IN": "Male life expectancy at birth (years)",
    "SE.ENR.SECO.FM.ZS": "Gender parity index, secondary school enrolment (female / male)",
    "SG.GEN.PARL.ZS": "Seats held by women in national parliament (%)",
    "SP.DYN.TFRT.IN": "Fertility rate (births per woman)",
    "SP.ADO.TFRT": "Adolescent fertility rate (births per 1,000 women aged 15-19)",
}
SHORT = {
    "SL.TLF.CACT.FE.ZS": "Female labour force participation",
    "SL.TLF.CACT.MA.ZS": "Male labour force participation",
    "SP.DYN.LE00.FE.IN": "Female life expectancy",
    "SP.DYN.LE00.MA.IN": "Male life expectancy",
    "SE.ENR.SECO.FM.ZS": "Secondary school parity index",
    "SG.GEN.PARL.ZS": "Women in parliament",
    "SP.DYN.TFRT.IN": "Fertility rate",
    "SP.ADO.TFRT": "Adolescent fertility",
}
# merge indicator with metada(topic)
series = pd.read_csv("Gender_StatsSeries.csv", usecols=["Series Code", "Topic"])
indicator_table = pd.DataFrame({"Code": list(INDICATORS.keys()),
                                "Indicator": list(INDICATORS.values())})
indicator_table = indicator_table.merge(series, left_on="Code", right_on="Series Code", how="left")
indicator_table = indicator_table[["Code", "Topic", "Indicator"]]
indicator_table

,Code,Topic,Indicator
0,SL.TLF.CACT.FE.ZS,Employment and Time Use,Female labour force participation rate (% of w...
1,SL.TLF.CACT.MA.ZS,Employment and Time Use,Male labour force participation rate (% of men...
2,SP.DYN.LE00.FE.IN,Population,Female life expectancy at birth (years)
3,SP.DYN.LE00.MA.IN,Population,Male life expectancy at birth (years)
4,SE.ENR.SECO.FM.ZS,Education,"Gender parity index, secondary school enrolmen..."
5,SG.GEN.PARL.ZS,Leadership,Seats held by women in national parliament (%)
6,SP.DYN.TFRT.IN,Population,Fertility rate (births per woman)
7,SP.ADO.TFRT,Health,"Adolescent fertility rate (births per 1,000 wo..."


## Data Cleaning
The zip file contains six CSV files. This notebook uses three of them:

| File | What it is | Why I need it |
|---|---|---|
| `Gender_StatsData.csv` | 262,615 rows of indicator values, one column per year (1960–2021) | the data itself |
| `Gender_StatsCountry.csv` | one row per economy: name, region, income group | to group countries and to spot regional aggregates |
| `Gender_StatsSeries.csv` | one row per indicator: definition, unit, source | to document the indicators |

The other three files (`...FootNote`, `...Country-Series`, `...Series-Time`) only contain notes about
where individual numbers came from. They are unuseful for the analysis, so I do not load them.

In [10]:
# a cheap look at the raw file (first 5 rows, all columns)
raw_peek = pd.read_csv("Gender_StatsData.csv", nrows=5)
print("Columns in the file:", raw_peek.shape[1])
raw_peek.iloc[0:4, 0:12]

Columns in the file: 67


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,1966,1967
0,Africa Eastern and Southern,AFE,A woman can apply for a passport in the same w...,SG.APL.PSPT.EQ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Africa Eastern and Southern,AFE,A woman can be head of household in the same w...,SG.HLD.HEAD.EQ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Africa Eastern and Southern,AFE,A woman can choose where to live in the same w...,SG.LOC.LIVE.EQ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Africa Eastern and Southern,AFE,A woman can get a job in the same way as a man...,SG.GET.JOBS.EQ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


The layout is now clear: the first 4 columns identify *who* (country) and *what* (indicator), and the
remaining 62 columns are years, with one empty artefact column at the very end. Each row is therefore one (country, indicator) pair, and the years run
across the row. It's a **wide** layout, and is difficulty for analysis, because adding a new year means adding a new column and every calculation has to be repeated once per year.

follow from the raw_peek I make a practical decision **only keep the years 1990–2021.** The file has 67 columns. Before 1990 most of the modern indicators are empty, I only needs the years 1990–2021 and the four identifier columns It makes pandas read only 36 columns, which keeps the notebook fast and the memory use modest.

In [13]:
YEARS = [str(year) for year in range(1990, 2022)]   
ID_COLUMNS = ["Country Name", "Country Code", "Indicator Name", "Indicator Code"]

raw = pd.read_csv("Gender_StatsData.csv", usecols=ID_COLUMNS + YEARS)

print("Shape of the subset:", raw.shape)
print("Countries in the file:", raw["Country Code"].nunique())#sum number of diffierent rows, avoid repetition
print("Indicators in the file:", raw["Indicator Code"].nunique())
raw.head(3)

Shape of the subset: (262615, 36)
Countries in the file: 265
Indicators in the file: 991


,Country Name,Country Code,Indicator Name,Indicator Code,1990,1991,1992,1993,1994,1995,...,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021
0,Africa Eastern and Southern,AFE,A woman can apply for a passport in the same w...,SG.APL.PSPT.EQ,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Africa Eastern and Southern,AFE,A woman can be head of household in the same w...,SG.HLD.HEAD.EQ,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Africa Eastern and Southern,AFE,A woman can choose where to live in the same w...,SG.LOC.LIVE.EQ,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
# only keep the eight chosen indicators
subset = raw[raw["Indicator Code"].isin(INDICATORS.keys())].copy()
print("Shape after filtering:", subset.shape)
print(subset["Indicator Code"].value_counts())

Shape after filtering: (2120, 36)
Indicator Code
SP.ADO.TFRT          265
SP.DYN.TFRT.IN       265
SL.TLF.CACT.FE.ZS    265
SL.TLF.CACT.MA.ZS    265
SP.DYN.LE00.FE.IN    265
SP.DYN.LE00.MA.IN    265
SG.GEN.PARL.ZS       265
SE.ENR.SECO.FM.ZS    265
Name: count, dtype: int64


per indicator has 265 rows(265 economies x 8 indicators = 2120)

## Exploratory Data Analysis
Programmatically explore the dataset. Tell the reader, how the dataset looks like and at least three most interesting observations that you can learn from the data.

## Descriptive Analytics
Interogate the dataset with descriptive analytics. Answer 3 questions that you find interesting, and that can be answered using descriptive analytics. Make each question a subsection of this section and follow the steps: 1. the question, 2. justify why it is interesting 3. the answer.

## Suggestions
Suggest further steps to enrich the dataset with additional (external) data, or apply some additional advanced modelling or analysis. You do not have to fully implement your suggestions, but you will have to explain and justify your suggestions.

## Bibliography
List all sources you have utilised in the making of this report here.